# Desarrollo del proceso ETL

Por Cristian Añon

## Red de Subtes de la Ciudad de Buenos Aires

En esta notebook detallo la construcción del script de extracción, transformación y carga de los distintos archivos _.csv_ que componen al dataset completo del período junio 2013 - junio 2026.

El principal desafío de este proceso, es poder crear un script lo suficientemente robusto para adaptarse a los diferentes formatos de encoding, fechas, wrapping de comillas, valores nulos, etc. de cada uno de los archivos.

Para abordar este problema, fueron necesarias muchas iteraciones de prueba y error para determinar con mayor exactitud las característicar de cada uno de los archivos. También fue necesario asegurar que los datos se puedan leer correctamente y sin ambigüedades en Power Bi. En este documento, solamente se detalla el proceso final en funcionamiento.

Se abordó el trabajo diviendo el total en dos conjuntos (2013-2021) y (2022-2026). A partir del 2022 la forma en que se registran los datos cambia sustancialmente:  

-Pasamos de tener 1 archivo por año, a tener entre 24 y 26.
-Las filas están envueltas en comillas.
-Cambian los encodings.
-Cambian los formatos de fecha.

Dividir en dos conjuntos este proceso me facilitó enormemente la tarea de aislar problemas y hacer un tratamiento adecuado. Si bien hice intentos por integrar todo el proceso en un sólo script, terminaba siendo muy complicado de leer y propenso a errores constantemente, no es práctico.

# Conjunto 2013-2021

Empiezo procesando únicamente los archivos correspondientes al período 2013-2021. Esta decisión surge a raíz de que los dataset del período contiguo (2022-2026) tienen cambios importantes en la forma de almacenar los registros de cada año. Más adelante se presenta su script correspondiente.

## Primera Etapa: estandarización y enriquecimiento

Empezamos definiendo las librerías:

In [34]:
import os

import pandas as pd
from charset_normalizer import from_bytes

Definimos rutas de entrada y de salida, y también la estructura final de columnas:

In [ ]:
folderpath = "../dataset/2013-2021/"
output_folder = "../output/13-21/"
os.makedirs(output_folder, exist_ok=True)
columnas_finales = ['FECHA', 'DESDE', 'HASTA', 'HORA', 'LINEA', 'ESTACION', 'BOCA', 'MOLINETE', 'MOLINETE_ORIGINAL', 'PAX_TOTAL']

Definimos el tamaño de la muestra que vamos a utilizar para detectar el encoding, y también seteamos el fallback:

In [36]:
ENCODING_SAMPLE_SIZE = 100_000
FALLBACK_ENCODING = 'latin-1'

Definimos la función encargada de detectar el encoding:

In [37]:
def detectar_encoding(filepath):
    """Detecta el encoding de un archivo CSV solamente sampleando los primeros 100.000 bytes."""
    with open(filepath, 'rb') as f:
        raw = f.read(ENCODING_SAMPLE_SIZE)
    resultado = from_bytes(raw).best()
    if resultado is not None:
        return resultado.encoding
    return FALLBACK_ENCODING

Definimos la función encargada de detectar el separador de cada archivo csv:

In [38]:
def detectar_separador(filepath, encoding):
    """Detecta el separador de un archivo CSV leyendo la primera línea. Verifica primero por ';' y por defecto usa ','."""
    with open(filepath, 'r', encoding=encoding, errors='replace') as f:
        header = f.readline()
    if ';' in header:
        return ';'
    return ','

Pasamos a definir la función que lee cada archivo csv. En caso de falla de lectura por haber detectado incorrectamente el encoding previamente, aplicamos el fallback:

In [39]:
def leer_csv_con_fallback(filepath, sep, encoding):
    """Lee un CSV con el encoding detectado y utiliza latin-1 como alternativa si ocurre un error de decodificación."""
    try:
        return pd.read_csv(filepath, sep=sep, engine='c', encoding=encoding, dtype=str, na_values=['NA', 'na', ''])
    except UnicodeDecodeError:
        print(f"  ⚠ Encoding '{encoding}' falló, reintentando con '{FALLBACK_ENCODING}'")
        return pd.read_csv(filepath, sep=sep, engine='c', encoding=FALLBACK_ENCODING, dtype=str, na_values=['NA', 'na', ''])

Definimos una función encargada exclusivamente de parsear correctamente las fechas. Este es un punto crítico en el procesamiento de los csv. Después de varias iteraciones, fuí detectando que había inconsistencias en los formatos de las fechas. Investigando encontré que la solución más robusta es hacer un "parseo en cascada", en dónde aquellas fechas que no fueron correctamente detectadas se van rellenando por descarte de combinaciones.

In [40]:
def parseo_en_cascada(serie):
    # Opción 1: ISO 8601 (sin ambigüedades)
    df_split = pd.to_datetime(serie, format='%Y-%m-%d', errors='coerce')

    # Opción 2: Día primero (ej. 1/6/2022 o 01/06/2022)
    df = df_split.fillna(pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce'))

    # Opción 3: Mes primero (ej. 06/01/2013) para los que fallan por día > 12 en mes
    df = df.fillna(pd.to_datetime(serie, format='%m/%d/%Y', errors='coerce'))

    # Opción 4: Fallback con dayfirst explícito para cualquier caso restante
    df = df.fillna(pd.to_datetime(serie, dayfirst=True, errors='coerce'))
    
    return df

La función "procesar_archivo" es la principal. Esta se encarga de llamar a todas las funciones anteriores, y además aplica la primer pasada de transformaciones:<br>

-Normaliza nombres de columnas.  
-Simplifica la columna "LINEA" a solamente mostrar la letra de cada línea (ej. "línea A" --> "A").  
-Normaliza el nombre de cada estación en mayúsculas.  
-Hace una copia de la columna "MOLINETE" para tener una referencia en caso de alguna inconsistencia.  
-Utilizando REGEX, extrae de la columna "MOLINETE" únicamente la parte correspondiente a la BOCA (ej. ALEM_N), y el molinete (ej. TURN02).  
-Da formato a la columna FECHA con pd.to_datetime  
-Da formato a la columna HORA como número entero (Int64).  
-Da formato a las columnas "DESDE" y "HASTA" como strings (esto es más práctico y seguro para exportar luego en Power Bi).  
-Da formato a la columna PAX_TOTAL como númeor intero (Int64).  
-Por último, extrae el año de cada archivo y guarda la primera pasada de normalización en archivos _.parquet_ individuales por cada año.<br>

Guardar archivos individuales por año, es una gran ventaja para poder aislar con mayor facilidad cualquier tipo de errores en el proceso. Además, es más amable con el uso de la memoria RAM, y agiliza la velocidad de procesamiento tanto en pandas como en Power Bi.

In [41]:
def procesar_archivo(filepath, archivo):
    print(f"Detectando encoding de: {archivo}")
    encoding = detectar_encoding(filepath)
    print(f"  Codec detectado: {encoding}")

    sep = detectar_separador(filepath, encoding)
    print(f"  Separador detectado: '{sep}'")

    print(f"  Leyendo {archivo}")
    df = leer_csv_con_fallback(filepath, sep, encoding)

    # --- Normalizar columnas ---
    df.columns = df.columns.str.strip().str.upper()
    df.rename(columns={'TOTAL': 'PAX_TOTAL'}, inplace=True)

    # --- Transformaciones de texto ---    
    df['LINEA'] = df['LINEA'].str.strip().str[-1]
    df['ESTACION'] = df['ESTACION'].str.strip().str.upper()

    df['MOLINETE_ORIGINAL'] = df['MOLINETE']

    partes = df['MOLINETE'].str.extract(r'^(?P<PREFIJO>[^_]+)_(?P<BOCA>.+)_(?P<MOLINETE>[^_]+)$')

    df['BOCA'] = partes['BOCA'].str.strip().str.upper()
    df['MOLINETE'] = partes['MOLINETE'].str.strip().str.upper()

    # --- Formatos Datetime --- 
    desde_dt = pd.to_datetime(df['DESDE'], format='mixed', errors='coerce')
    hasta_dt = pd.to_datetime(df['HASTA'], format='mixed', errors='coerce')

    df['HORA'] = desde_dt.dt.hour.astype('Int64') # Int64 maneja valores nulos si falla el parseo
    df['DESDE'] = desde_dt.dt.strftime('%H:%M')
    df['HASTA'] = hasta_dt.dt.strftime('%H:%M')

    df['FECHA'] = parseo_en_cascada(df['FECHA'])

    df['PAX_TOTAL'] = pd.to_numeric(df['PAX_TOTAL'], errors='coerce').astype('Int64')

    # --- Seleccionar y guardar ---
    df = df[columnas_finales]
    numero = ''.join(filter(str.isdigit, archivo))[-4:]
    nombre_salida = f"df{numero}.parquet"
    ruta_salida = os.path.join(output_folder, nombre_salida)
    df.to_parquet(
    ruta_salida,
    index=False,
    engine='pyarrow',
    compression='snappy',
    coerce_timestamps='ms'
)
    print(f"  Guardado en: {ruta_salida}")

Por último, el loop principal que incia el proceso:

In [42]:
# --- Loop principal ---
for archivo in sorted(os.listdir(folderpath)):
    if archivo.endswith('.csv'):
        filepath = os.path.join(folderpath, archivo)
        procesar_archivo(filepath, archivo)

Detectando encoding de: molinetes_2013-junio_dic.csv
  Codec detectado: ascii
  Separador detectado: ','
  Leyendo molinetes_2013-junio_dic.csv
  ⚠ Encoding 'ascii' falló, reintentando con 'latin-1'
  Guardado en: ./output/13-21/df2013.parquet
Detectando encoding de: molinetes_2014.csv
  Codec detectado: ascii
  Separador detectado: ','
  Leyendo molinetes_2014.csv
  Guardado en: ./output/13-21/df2014.parquet
Detectando encoding de: molinetes_2015.csv
  Codec detectado: utf_8
  Separador detectado: ','
  Leyendo molinetes_2015.csv
  Guardado en: ./output/13-21/df2015.parquet
Detectando encoding de: molinetes_2016.csv
  Codec detectado: ascii
  Separador detectado: ','
  Leyendo molinetes_2016.csv
  ⚠ Encoding 'ascii' falló, reintentando con 'latin-1'
  Guardado en: ./output/13-21/df2016.parquet
Detectando encoding de: molinetes_2017.csv
  Codec detectado: ascii
  Separador detectado: ','
  Leyendo molinetes_2017.csv
  ⚠ Encoding 'ascii' falló, reintentando con 'latin-1'
  Guardado en: 

## Segunda Etapa: detección de nulos

#### Buscamos filas completamente vacías: si existen las eliminamos, y volvemos a guardar los archivos.

In [ ]:
folderpath = "../output/13-21/"
output_folder = "../output/13-21" # usamos la misma ruta para directamente sobreescribir los archivos procesados
sum = 0
os.makedirs(output_folder, exist_ok=True)

for archivo in sorted(os.listdir(folderpath)):
    if archivo.endswith('.parquet'):
        filepath = os.path.join(folderpath, archivo)
        output_filepath = os.path.join(output_folder, archivo)
        
        print(f"\nProcesando: {archivo}")
        
        # 1. Lectura del archivo individual
        df = pd.read_parquet(filepath, engine='pyarrow')
        
        # 2. Verificación y limpieza de filas completamente vacías
        filas_nulas = df.isna().all(axis=1).sum()
        if filas_nulas > 0:
            print(f"  - Se detectaron {filas_nulas} filas completamente nulas. Eliminando...")
            df.dropna(inplace=True, how='all', axis=0)
            print("  - Filas eliminadas con éxito.")
            sum += filas_nulas
        else:
            print("  - No se detectaron filas completamente nulas.")
            
        # 3. Guardado en la carpeta de destino con compresión snappy
        df.to_parquet(
            output_filepath,
            index=False,
            engine='pyarrow',
            compression='snappy',
            coerce_timestamps='ms'
        )
        print(f"  - Guardado en: {output_filepath}")
print(f"\nTotal de filas completamente nulas eliminadas en todos los archivos: {sum}")


Procesando: df2013.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/13-21\df2013.parquet

Procesando: df2014.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/13-21\df2014.parquet

Procesando: df2015.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/13-21\df2015.parquet

Procesando: df2016.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/13-21\df2016.parquet

Procesando: df2017.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/13-21\df2017.parquet

Procesando: df2018.parquet
  - Se detectaron 78207 filas completamente nulas. Eliminando...
  - Filas eliminadas con éxito.
  - Guardado en: ./output/13-21\df2018.parquet

Procesando: df2019.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/13-21\df2019.parquet

Procesando: df2020.parquet
  - Se detectaron 1836 filas completamente nulas. Eliminando..

### Chequeamos si hay filas con algún valor nulo

In [44]:
folderpath = "./output/13-21"
lista_nulos = []

for archivo in sorted(os.listdir(folderpath)):
    if archivo.endswith('.parquet'):
        filepath = os.path.join(folderpath, archivo)
        
        print(f"\nProcesando: {archivo}")
        
        # 1. Lectura del archivo individual
        df = pd.read_parquet(filepath, engine='pyarrow')
        
        # 2. Verificación y limpieza de filas con valores nulos
        filas_nulas = df.isna().any(axis=1).sum()
        if filas_nulas > 0:
            print(f"  - Se detectaron {filas_nulas} filas con valores nulos.")
            lista_nulos.append(archivo)
        else:
            print("  - No se detectaron filas con valores nulos.")
print(f"\nLos archivos con filas nulas son: {lista_nulos}")


Procesando: df2013.parquet
  - Se detectaron 221 filas con valores nulos.

Procesando: df2014.parquet
  - No se detectaron filas con valores nulos.

Procesando: df2015.parquet
  - No se detectaron filas con valores nulos.

Procesando: df2016.parquet
  - No se detectaron filas con valores nulos.

Procesando: df2017.parquet
  - Se detectaron 27 filas con valores nulos.

Procesando: df2018.parquet
  - Se detectaron 56 filas con valores nulos.

Procesando: df2019.parquet
  - No se detectaron filas con valores nulos.

Procesando: df2020.parquet
  - No se detectaron filas con valores nulos.

Procesando: df2021.parquet
  - No se detectaron filas con valores nulos.

Los archivos con filas nulas son: ['df2013.parquet', 'df2017.parquet', 'df2018.parquet']


### Revisamos uno por uno los archivos para ver qué valores faltan

#### 2013

In [ ]:
df = pd.read_parquet("../output/13-21/df2013.parquet", engine='pyarrow')
df.isna().sum()

FECHA                  0
DESDE                  0
HASTA                  0
HORA                   0
LINEA                  0
ESTACION               0
BOCA                 221
MOLINETE             221
MOLINETE_ORIGINAL      0
PAX_TOTAL              0
dtype: int64

Revisamos que pasa con las columnas con nulls

In [46]:
df[df['BOCA'].isna()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
425249,2013-08-05,08:30,08:44,8,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,11
425253,2013-08-05,08:45,08:59,8,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,20
425257,2013-08-05,09:00,09:14,9,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,13
425261,2013-08-05,09:15,09:29,9,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,18
425265,2013-08-05,09:30,09:44,9,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,13
...,...,...,...,...,...,...,...,...,...,...
426847,2013-08-12,10:00,10:14,10,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,21
426851,2013-08-12,10:15,10:29,10,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,23
426855,2013-08-12,10:30,10:44,10,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,24
426859,2013-08-12,10:45,10:59,10,E,AVENIDA LA PLATA,None,None,Bonifacio_Tur03,30


In [47]:
df[df['BOCA'].isna()]['MOLINETE_ORIGINAL'].value_counts()

MOLINETE_ORIGINAL
Bonifacio_Tur03    221
Name: count, dtype: int64

Las 221 filas con valores nulos son del mismo "MOLINETE ORIGINAL".
Vemos que no se extrajeron los datos de "BOCA" y "MOLINETE". Extraemos correctamente y reemplazamos nulls.

In [48]:
df['BOCA'].fillna('Bonifacio', inplace=True)
df['MOLINETE'].fillna('Tur03', inplace=True)

In [49]:
df.isna().sum()

FECHA                0
DESDE                0
HASTA                0
HORA                 0
LINEA                0
ESTACION             0
BOCA                 0
MOLINETE             0
MOLINETE_ORIGINAL    0
PAX_TOTAL            0
dtype: int64

Guardamos el archivo limpio.

In [ ]:
df.to_parquet("../output/13-21/df2013.parquet", engine='pyarrow', index=False, compression='snappy', coerce_timestamps='ms')

#### 2017

In [ ]:
df = pd.read_parquet("../output/13-21/df2017.parquet", engine='pyarrow')
df.isna().sum()

FECHA                 0
DESDE                 0
HASTA                 0
HORA                  0
LINEA                 0
ESTACION              0
BOCA                 27
MOLINETE             27
MOLINETE_ORIGINAL     0
PAX_TOTAL             0
dtype: int64

In [52]:
df[df['BOCA'].isna()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
5398215,2017-10-02,10:00,10:15,10,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5398225,2017-10-02,10:30,10:45,10,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5398821,2017-10-04,07:15,07:30,7,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5398848,2017-10-04,08:30,08:45,8,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5398859,2017-10-04,09:00,09:15,9,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5399509,2017-10-06,07:30,07:45,7,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5400505,2017-10-09,08:00,08:15,8,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5400845,2017-10-10,07:15,07:30,7,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5401586,2017-10-12,09:15,09:30,9,H,HOSPITALES,None,None,BONIFACIO_TUR02,0
5401616,2017-10-12,10:45,11:00,10,H,HOSPITALES,None,None,BONIFACIO_TUR02,0


Notamos que no existe la estación BOCA "BONIFACIO" para la estación HOSPITALES. Además, todos los registros tiene 0 pasajeros. Probablemente se trate de un error o mantenimiento. Podemos eliminar.

In [53]:
df.dropna(inplace=True)
df.isna().sum()

FECHA                0
DESDE                0
HASTA                0
HORA                 0
LINEA                0
ESTACION             0
BOCA                 0
MOLINETE             0
MOLINETE_ORIGINAL    0
PAX_TOTAL            0
dtype: int64

Guardamos

In [ ]:
df.to_parquet("../output/13-21/df2017.parquet", engine='pyarrow', index=False, compression='snappy', coerce_timestamps='ms')

#### 2018

In [ ]:
df = pd.read_parquet("../output/13-21/df2018.parquet", engine='pyarrow')
df.isna().sum()

FECHA                 0
DESDE                 0
HASTA                 0
HORA                  0
LINEA                56
ESTACION              0
BOCA                 56
MOLINETE             56
MOLINETE_ORIGINAL     0
PAX_TOTAL             0
dtype: int64

In [56]:
df[df['LINEA'].isna()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
3024981,2018-04-03,08:30,08:45,8,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3028373,2018-04-03,10:00,10:15,10,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3030167,2018-04-03,10:45,11:00,10,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3066428,2018-04-04,10:45,11:00,10,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3096746,2018-04-05,07:30,07:45,7,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3100421,2018-04-05,09:15,09:30,9,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3136896,2018-04-06,09:00,09:15,9,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3137427,2018-04-06,09:15,09:30,9,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3140114,2018-04-06,10:30,10:45,10,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0
3163075,2018-04-07,07:00,07:15,7,None,TALLER BONIFACIO,None,None,Bonifacio_Tur02,0


Vemos que la estación es "TALLER BONIFACIO" y las filas tienen 0 registros. Eliminamos.

In [57]:
df.dropna(inplace=True)
df.isna().sum()

FECHA                0
DESDE                0
HASTA                0
HORA                 0
LINEA                0
ESTACION             0
BOCA                 0
MOLINETE             0
MOLINETE_ORIGINAL    0
PAX_TOTAL            0
dtype: int64

Guardamos

In [ ]:
df.to_parquet("../output/13-21/df2018.parquet", engine='pyarrow', index=False, compression='snappy', coerce_timestamps='ms')

## Tercera etapa: tratamiento de nulos y normalización de estaciones, bocas, y molinetes

Acá vamos a revisar en profundidad todo el conjunto 2013-2021. Vamos linea por linea. Buscamos inconsistencias en los nombres, filas incorrectas, caracteres extraños, estaciones incorrectas, filas de prueba, etc. Este paso es crítico para tener un dataset bien prolijo.

También rellenamos valores nulos según el tipo de dato. Se detallará a continuación cada caso.

### Concatenamos en un solo df en memoria para diagnóstico general

Concatenar en un sólo _dataframe_ nos permite trabajar con mayor comodidad.

In [ ]:
df_unificado = pd.read_parquet("../output/13-21", engine='pyarrow')
df_unificado['FECHA'].min(), df_unificado['FECHA'].max()

(Timestamp('2013-06-01 00:00:00'), Timestamp('2021-12-31 00:00:00'))

In [60]:
df_unificado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90008149 entries, 0 to 90008148
Data columns (total 10 columns):
 #   Column             Dtype         
---  ------             -----         
 0   FECHA              datetime64[ms]
 1   DESDE              object        
 2   HASTA              object        
 3   HORA               Int64         
 4   LINEA              object        
 5   ESTACION           object        
 6   BOCA               object        
 7   MOLINETE           object        
 8   MOLINETE_ORIGINAL  object        
 9   PAX_TOTAL          Int64         
dtypes: Int64(2), datetime64[ms](1), object(7)
memory usage: 6.9+ GB


### A

Estaciones

In [61]:
df_unificado[df_unificado['LINEA'] == 'A']['ESTACION'].unique()

array(['ACOYTE', 'ALBERTI', 'CARABOBO', 'CASTRO BARROS', 'CONGRESO',
       'FLORES', 'LIMA', 'LORIA', 'PASCO', 'PERU', 'PIEDRAS',
       'PLAZA DE MAYO', 'PLAZA MISERERE', 'PRIMERA JUNTA', 'PUAN',
       'RIO DE JANEIRO', 'SAENZ PEÑA', 'SAN PEDRITO', 'SAENZ PE<D1>A',
       'SAENZ PEÃ\x83Â±A', 'SAENZ PEÃ±A', 'SAENZ PE�A', 'SAENZ PEÏ¿½A',
       'SAENZ PEÂ±A', 'SAENZ PEÂ¤A'], dtype=object)

In [62]:
df_unificado.loc[
    (df_unificado['LINEA'] == 'A') & 
    (df_unificado['ESTACION'].str.contains('SAENZ', case=False, na=False)),
    'ESTACION'
] = 'SAENZ PEÑA'

In [63]:
df_unificado[df_unificado['LINEA'] == 'A']['ESTACION'].unique()

array(['ACOYTE', 'ALBERTI', 'CARABOBO', 'CASTRO BARROS', 'CONGRESO',
       'FLORES', 'LIMA', 'LORIA', 'PASCO', 'PERU', 'PIEDRAS',
       'PLAZA DE MAYO', 'PLAZA MISERERE', 'PRIMERA JUNTA', 'PUAN',
       'RIO DE JANEIRO', 'SAENZ PEÑA', 'SAN PEDRITO'], dtype=object)

Bocas

In [64]:
df_unificado[df_unificado['LINEA'] == 'A']['BOCA'].unique()

array(['A_ACOYTE_N', 'A_ACOYTE_S', 'A_ALBERTI', 'A_CARABOBO_E',
       'A_CARABOBO_O', 'A_CBARROS_N', 'A_CBARROS_S', 'A_CONGRESO_N',
       'A_CONGRESO_S', 'A_FLORES_OESTE', 'A_FLORES_ESTE', 'A_LIMA_S',
       'A_LIMA_N', 'A_LORIA_N', 'A_LORIA_S', 'A_PASCO', 'A_PERU_S',
       'A_PERU_N', 'A_PIEDRAS_S', 'A_PIEDRAS_N', 'A_PMAYO_O', 'A_PMAYO_E',
       'A_MISERERE_Q_HALL', 'A_MISERERE_S', 'A_MISERERE_Q_NE',
       'MISERERE_Q_NO', 'A_PJUNTA_S', 'PJUNTA_S', 'A_PJUNTA_N',
       'A_PUAN_O', 'A_PUAN_E', 'PUAN_E', 'PUAN_O', 'A_RJANEIRO_N',
       'A_RJANEIRO_S', 'A_SNZPENA_S', 'A_SNZPENA_N', 'A_SANPEDRITO_ESTE',
       'A_SANPEDRITO_OESTE', 'A_SNZPEÑA_S', 'A_SNZPEÑA_N', 'ACOYTE_N',
       'CBARROS_N', 'LIMA_N', 'MISERERE_Q_NE', 'PERU_S',
       'SANPEDRITO_ESTE', 'SNZPENA_N', 'SNZPENA_S', 'CARABOBO_E',
       'CONGRESO_N', 'LORIA_S', 'CONGRESO_S', 'LORIA_N',
       'MISERERE_Q_HALL', 'PIEDRAS_S', 'RJANEIRO_S', 'CBARROS_S',
       'FLORES_OESTE', 'PASCO', 'RJANEIRO_N', 'SANPEDRITO_OESTE',
   

In [65]:
df_unificado['BOCA'] = df_unificado['BOCA'].str.replace(r'^A_', '', regex=True)

In [66]:
df_unificado['BOCA'] = df_unificado['BOCA'].str.replace(r'^SNZPENA_(.)', r'SNZPEÑA_\1', regex=True)

In [67]:
df_unificado[df_unificado['LINEA'] == 'A']['BOCA'].unique()

array(['ACOYTE_N', 'ACOYTE_S', 'ALBERTI', 'CARABOBO_E', 'CARABOBO_O',
       'CBARROS_N', 'CBARROS_S', 'CONGRESO_N', 'CONGRESO_S',
       'FLORES_OESTE', 'FLORES_ESTE', 'LIMA_S', 'LIMA_N', 'LORIA_N',
       'LORIA_S', 'PASCO', 'PERU_S', 'PERU_N', 'PIEDRAS_S', 'PIEDRAS_N',
       'PMAYO_O', 'PMAYO_E', 'MISERERE_Q_HALL', 'MISERERE_S',
       'MISERERE_Q_NE', 'MISERERE_Q_NO', 'PJUNTA_S', 'PJUNTA_N', 'PUAN_O',
       'PUAN_E', 'RJANEIRO_N', 'RJANEIRO_S', 'SNZPEÑA_S', 'SNZPEÑA_N',
       'SANPEDRITO_ESTE', 'SANPEDRITO_OESTE', 'MISERERE_NE'], dtype=object)

Molinetes: notese que esto incluye a los ascensores y molinetes especiales.

In [68]:
df_unificado[df_unificado['LINEA'] == "A"]['MOLINETE'].unique()

array(['TURN06', 'TURN05', 'TURN01', 'TURN04', 'TURN03', 'TURN02',
       'ASC01', 'DISCAP04', 'TURN07', 'TURN08'], dtype=object)

### B

Estaciones

In [69]:
df_unificado[df_unificado['LINEA'] == 'B']['ESTACION'].unique()

array(['ANGEL GALLARDO', 'CALLAO', 'CARLOS GARDEL', 'CARLOS PELLEGRINI',
       'DORREGO', 'ECHEVERRIA', 'FEDERICO LACROZE', 'FLORIDA',
       'LEANDRO N. ALEM', 'LOS INCAS', 'MALABIA', 'MEDRANO', 'PASTEUR',
       'PUEYRREDON', 'ROSAS', 'TRONADOR', 'URUGUAY', 'CALLAO.B'],
      dtype=object)

In [70]:
df_unificado.loc[df_unificado['LINEA'] == 'B', 'ESTACION'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'B', 'ESTACION'].replace("CALLAO.B", "CALLAO")
)

In [71]:
df_unificado[df_unificado['LINEA'] == 'B']['ESTACION'].unique()

array(['ANGEL GALLARDO', 'CALLAO', 'CARLOS GARDEL', 'CARLOS PELLEGRINI',
       'DORREGO', 'ECHEVERRIA', 'FEDERICO LACROZE', 'FLORIDA',
       'LEANDRO N. ALEM', 'LOS INCAS', 'MALABIA', 'MEDRANO', 'PASTEUR',
       'PUEYRREDON', 'ROSAS', 'TRONADOR', 'URUGUAY'], dtype=object)

Bocas

In [72]:
df_unificado[df_unificado['LINEA'] == 'B']['BOCA'].unique()

array(['B_GALLARDO_N', 'B_GALLARDO_S', 'B_CALLAOB_N', 'B_CALLAOB_S',
       'B_GARDEL_N', 'B_GARDEL_S', 'B_PELLEGRINI_E', 'B_PELLEGRINI_O',
       'B_DORREGO_N', 'B_DORREGO_S', 'B_ECHEVERRIA_OESTE',
       'B_ECHEVERRIA_ESTE', 'B_LACROZE_O', 'B_LACROZE_E', 'B_LACROZE_S',
       'B_FLORIDA_O', 'B_FLORIDA_E', 'B_ALEM_N', 'B_ALEM_S', 'B_LOSINCAS',
       'B_MALABIA_N', 'B_MALABIA_S', 'B_MEDRANO_N', 'B_MEDRANO_S',
       'B_PASTEUR_S', 'B_PASTEUR_N', 'B_PUEYR_N', 'B_PUEYR_S',
       'B_JMROSAS_ESTE', 'B_JMROSAS_OESTE', 'B_TRONADOR', 'B_URUGUAY_N',
       'B_URUGUAY_S', 'ECHEVERRIA_ESTE', 'ECHEVERRIA_OESTE', 'ALEM_S',
       'FLORIDA_O', 'GALLARDO_S', 'LACROZE_E', 'LOSINCAS', 'PASTEUR_S',
       'TRONADOR', 'GARDEL_S', 'LACROZE_O', 'MALABIA_N', 'MEDRANO_N',
       'MEDRANO_S', 'PELLEGRINI_E', 'ALEM_N', 'LACROZE_S', 'CALLAOB_N',
       'GALLARDO_N', 'JMROSAS_ESTE', 'JMROSAS_OESTE', 'PUEYR_N',
       'URUGUAY_S', 'CALLAOB_S', 'DORREGO_N', 'GARDEL_N', 'PUEYR_S',
       'URUGUAY_N', 'PASTEUR_N'

In [73]:
df_unificado.loc[df_unificado['LINEA'] == 'B', 'BOCA'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'B', 'BOCA'].str.replace(r'^B_', '', regex=True)
)

In [74]:
df_unificado[df_unificado['LINEA'] == 'B']['BOCA'].unique()

array(['GALLARDO_N', 'GALLARDO_S', 'CALLAOB_N', 'CALLAOB_S', 'GARDEL_N',
       'GARDEL_S', 'PELLEGRINI_E', 'PELLEGRINI_O', 'DORREGO_N',
       'DORREGO_S', 'ECHEVERRIA_OESTE', 'ECHEVERRIA_ESTE', 'LACROZE_O',
       'LACROZE_E', 'LACROZE_S', 'FLORIDA_O', 'FLORIDA_E', 'ALEM_N',
       'ALEM_S', 'LOSINCAS', 'MALABIA_N', 'MALABIA_S', 'MEDRANO_N',
       'MEDRANO_S', 'PASTEUR_S', 'PASTEUR_N', 'PUEYR_N', 'PUEYR_S',
       'JMROSAS_ESTE', 'JMROSAS_OESTE', 'TRONADOR', 'URUGUAY_N',
       'URUGUAY_S'], dtype=object)

Molinetes

In [75]:
df_unificado[df_unificado['LINEA'] == 'B']['MOLINETE'].unique()

array(['TURN02', 'TURN03', 'TURN01', 'TURN04', 'TURN05', 'TURN06',
       'TURN07', 'TURN09', 'TURN08', 'ASC01'], dtype=object)

### C

Estaciones

In [76]:
df_unificado[df_unificado['LINEA'] == 'C']['ESTACION'].unique()

array(['AVENIDA DE MAYO', 'CONSTITUCION', 'DIAGONAL NORTE',
       'GENERAL SAN MARTIN', 'INDEPENDENCIA', 'LAVALLE', 'MARIANO MORENO',
       'RETIRO', 'SAN JUAN'], dtype=object)

Bocas

In [77]:
df_unificado[df_unificado['LINEA'] == 'C']['BOCA'].unique()

array(['C_AVMAYO_S', 'C_AVMAYO_N', 'C_CONSTITUCION', 'CONSTITUCION',
       'C_DNORTE_S', 'C_DNORTE_N', 'C_SANMARTIN_N', 'C_SANMARTIN_S',
       'C_INDEPEN', 'C_LAVALLE_S', 'C_LAVALLE_N', 'C_MORENO_N',
       'C_MORENO_S', 'C_RETIRO', 'C_SANJUAN', 'INDEPEN', 'LAVALLE_S',
       'RETIRO', 'AVMAYO_S', 'MORENO_S', 'SANJUAN', 'SANMARTIN_N',
       'DNORTE_S', 'MORENO_N', 'SANMARTIN_S', 'AVMAYO_N', 'DNORTE_N',
       'LAVALLE_N', 'CONSTITUCION_PLAZA'], dtype=object)

Limpiamos las denominaciones de línea, no las necesitamos y mete ruido.

In [78]:
df_unificado.loc[df_unificado['LINEA'] == 'C', 'BOCA'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'C', 'BOCA'].str.replace(r'^C_', '', regex=True)
)

In [79]:
df_unificado[df_unificado['LINEA'] == 'C']['BOCA'].unique()

array(['AVMAYO_S', 'AVMAYO_N', 'CONSTITUCION', 'DNORTE_S', 'DNORTE_N',
       'SANMARTIN_N', 'SANMARTIN_S', 'INDEPEN', 'LAVALLE_S', 'LAVALLE_N',
       'MORENO_N', 'MORENO_S', 'RETIRO', 'SANJUAN', 'CONSTITUCION_PLAZA'],
      dtype=object)

In [80]:
df_unificado[df_unificado['LINEA'] == 'C']['MOLINETE'].unique()

array(['TURN02', 'TURN03', 'TURN01', 'TURN25', 'TURN27', 'TURN28',
       'TURN29', 'TURN30', 'TURN31', 'TURN32', 'TURN23', 'TURN24',
       'TURN26', 'TURN09', 'TURN10', 'TURN11', 'TURN12', 'TURN13',
       'TURN14', 'TURN15', 'TURN16', 'TURN17', 'TURN18', 'TURN05',
       'TURN08', 'TURN19', 'TURN20', 'TURN22', 'TURN21', 'TURN06',
       'TURN07', 'TURN04', 'TURN1', 'TURN0', 'TURN', 'ASC01'],
      dtype=object)

### D

Estaciones

In [81]:
df_unificado[df_unificado['LINEA'] == 'D']['ESTACION'].unique()

array(['9 DE JULIO', 'AGUERO', 'BULNES', 'CALLAO.', 'CATEDRAL',
       'CONGRESO DE TUCUMAN', 'FACULTAD DE MEDICINA', 'JOSE HERNANDEZ',
       'JURAMENTO', 'MINISTRO CARRANZA', 'OLLEROS', 'PALERMO',
       'PLAZA ITALIA', 'PUEYRREDON.', 'SCALABRINI ORTIZ', 'TRIBUNALES',
       'CALLAO', 'PUEYRREDON.D', 'AGÜERO', 'PUEYRREDON', 'AGÃ\x83Â¼ERO',
       'AGÃ¼ERO', 'AG�ERO', 'AGÂ³ERO', 'AGÏ¿½ERO', 'AGÂ\x81ERO'],
      dtype=object)

In [82]:
df_unificado.loc[df_unificado['LINEA'] == 'D', 'ESTACION'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'D', 'ESTACION'].replace({
        "PUEYRREDON.":"PUEYRREDON",
        "PUEYRREDON.D": "PUEYRREDON",
        "CALLAO.":"CALLAO"
    })
)

In [83]:
mask = (df_unificado['LINEA'] == 'D') & (df_unificado['ESTACION'].str.match(r'^AG.*ERO$', na=False))

df_unificado.loc[mask, 'ESTACION'] = 'AGÜERO'

In [84]:
df_unificado[df_unificado['LINEA'] == 'D']['ESTACION'].unique()

array(['9 DE JULIO', 'AGÜERO', 'BULNES', 'CALLAO', 'CATEDRAL',
       'CONGRESO DE TUCUMAN', 'FACULTAD DE MEDICINA', 'JOSE HERNANDEZ',
       'JURAMENTO', 'MINISTRO CARRANZA', 'OLLEROS', 'PALERMO',
       'PLAZA ITALIA', 'PUEYRREDON', 'SCALABRINI ORTIZ', 'TRIBUNALES'],
      dtype=object)

Bocas

In [85]:
df_unificado[df_unificado['LINEA'] == 'D']['BOCA'].unique()

array(['D_9JULIO_S', 'D_9JULIO_N', 'D_9JULIO_SO', 'D_AGUERO',
       'D_BULNES_N', 'D_BULNES_S', 'D_CALLAOD_S', 'D_CALLAOD_N',
       'D_CALLAOD_E', 'D_CATEDRAL_E', 'D_CATEDRAL_O', 'D_CONGRESOTUC_N',
       'D_CONGRESOTUC_O', 'D_CONGRESOTUC_C', 'D_CONGRESOTUC_S',
       'D_FMEDICINA_N', 'D_FMEDICINA_S', 'D_J_HERNANDEZ_ESTE',
       'D_J_HERNANDEZ_OESTE', 'D_JURAMENTO_ESTE', 'D_JURAMENTO_OESTE',
       'D_CARRANZA', 'D_OLLEROS_ESTE', 'D_OLLEROS_OESTE', 'D_PALERMO',
       'D_PITALIA_O', 'D_PITALIA_E', 'D_PUEYRREDON', 'D_SCAL_ORTIZ_NORTE',
       'D_SCAL_ORTIZ_SUR', 'D_TRIBUNA_O', 'D_TRIBUNA_E', 'BULNES_N',
       'CONGRESOTUC_O', 'FMEDICINA_N', 'J_HERNANDEZ_OESTE',
       'JURAMENTO_ESTE', 'JURAMENTO_OESTE', 'OLLEROS_OESTE', 'PALERMO',
       'PITALIA_O', 'SCAL_ORTIZ_NORTE', 'TRIBUNA_O', '9JULIO_S', 'AGUERO',
       'CALLAOD_S', 'CONGRESOTUC_C', 'CONGRESOTUC_N', 'J_HERNANDEZ_ESTE',
       'OLLEROS_ESTE', 'BULNES_S', 'CARRANZA', 'CATEDRAL_O',
       'FMEDICINA_S', 'PUEYRREDON', 'SCAL_ORT

In [86]:
df_unificado.loc[df_unificado['LINEA'] == 'D', 'BOCA'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'D', 'BOCA'].str.replace(r'^D_', '', regex=True)
)

In [87]:
df_unificado[df_unificado['LINEA'] == 'D']['BOCA'].unique()

array(['9JULIO_S', '9JULIO_N', '9JULIO_SO', 'AGUERO', 'BULNES_N',
       'BULNES_S', 'CALLAOD_S', 'CALLAOD_N', 'CALLAOD_E', 'CATEDRAL_E',
       'CATEDRAL_O', 'CONGRESOTUC_N', 'CONGRESOTUC_O', 'CONGRESOTUC_C',
       'CONGRESOTUC_S', 'FMEDICINA_N', 'FMEDICINA_S', 'J_HERNANDEZ_ESTE',
       'J_HERNANDEZ_OESTE', 'JURAMENTO_ESTE', 'JURAMENTO_OESTE',
       'CARRANZA', 'OLLEROS_ESTE', 'OLLEROS_OESTE', 'PALERMO',
       'PITALIA_O', 'PITALIA_E', 'PUEYRREDON', 'SCAL_ORTIZ_NORTE',
       'SCAL_ORTIZ_SUR', 'TRIBUNA_O', 'TRIBUNA_E', 'PUEYRREDON_ALIV'],
      dtype=object)

Molinetes

In [88]:
df_unificado[df_unificado['LINEA'] == 'D']['MOLINETE'].unique()

array(['TURN01', 'TURN02', 'TURN03', 'TURN04', 'ASC01', 'TURN05',
       'TURN07', 'TURN08', 'TURN06', 'TURN0'], dtype=object)

### E

Estaciones

In [89]:
df_unificado[df_unificado['LINEA'] == 'E']['ESTACION'].unique()

array(['AVENIDA LA PLATA', 'BOEDO', 'BOLIVAR', 'EMILIO MITRE',
       'ENTRE RIOS', 'GENERAL BELGRANO', 'INDEPENDENCIA.',
       'JOSE MARIA MORENO', 'JUJUY', 'MEDALLA MILAGROSA', 'PICHINCHA',
       'PZA. DE LOS VIRREYES', 'SAN JOSE', 'URQUIZA', 'VARELA',
       'INDEPENDENCIA.H', 'INDEPENDENCIA', 'CORREO CENTRAL', 'RETIRO E',
       'CATALINAS'], dtype=object)

In [90]:
df_unificado.loc[df_unificado['LINEA'] == 'E', 'ESTACION'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'E', 'ESTACION'].replace({
        "INDEPENDENCIA.H": "INDEPENDENCIA",
        "INDEPENDENCIA.": "INDEPENDENCIA"})
)

In [91]:
df_unificado[df_unificado['LINEA'] == 'E']['ESTACION'].unique()

array(['AVENIDA LA PLATA', 'BOEDO', 'BOLIVAR', 'EMILIO MITRE',
       'ENTRE RIOS', 'GENERAL BELGRANO', 'INDEPENDENCIA',
       'JOSE MARIA MORENO', 'JUJUY', 'MEDALLA MILAGROSA', 'PICHINCHA',
       'PZA. DE LOS VIRREYES', 'SAN JOSE', 'URQUIZA', 'VARELA',
       'CORREO CENTRAL', 'RETIRO E', 'CATALINAS'], dtype=object)

Bocas

In [92]:
df_unificado[df_unificado['LINEA'] == 'E']['BOCA'].unique()

array(['E_LAPLATA', 'Bonifacio', 'E_BOEDO', 'E_BOLIVAR_N', 'E_BOLIVAR_S',
       'E_EMITRE', 'E_ERIOS', 'E_BELGRANO', 'E_INDEPEN', 'E_MORENO',
       'E_JUJUY', 'E_MEDALLA', 'E_PICHIN', 'E_VIRREYES', 'E_SANJOSE',
       'E_URQUIZA', 'E_VARELA', 'VIRREYES_O', 'E_VIRREYES_E', 'INDEPEN',
       'SANJOSE', 'URQUIZA', 'VIRREYES_E', 'BOEDO', 'BOLIVAR_N',
       'VIRREYES', 'PICHIN', 'EMITRE', 'JUJUY', 'LAPLATA', 'VARELA',
       'MORENO', 'MEDALLA', 'ERIOS', 'BELGRANO', 'BOLIVAR_S',
       'CCENTRAL_N', 'RETIROE_N', 'RETIROE_S', 'CATALINAS_N',
       'CCENTRAL_S', 'CATALINAS_S'], dtype=object)

In [93]:
df_unificado.loc[df_unificado['LINEA'] == 'E', 'BOCA'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'E', 'BOCA'].str.replace(r'^E_', '', regex=True)
)

In [94]:
df_unificado[df_unificado['LINEA'] == 'E']['BOCA'].unique()

array(['LAPLATA', 'Bonifacio', 'BOEDO', 'BOLIVAR_N', 'BOLIVAR_S',
       'EMITRE', 'ERIOS', 'BELGRANO', 'INDEPEN', 'MORENO', 'JUJUY',
       'MEDALLA', 'PICHIN', 'VIRREYES', 'SANJOSE', 'URQUIZA', 'VARELA',
       'VIRREYES_O', 'VIRREYES_E', 'CCENTRAL_N', 'RETIROE_N', 'RETIROE_S',
       'CATALINAS_N', 'CCENTRAL_S', 'CATALINAS_S'], dtype=object)

Molinetes

In [95]:
df_unificado[df_unificado['LINEA'] == 'E']['MOLINETE'].unique()

array(['TURN02', 'TURN03', 'TURN01', 'Tur03', 'TURN05', 'TURN04',
       'TURN06', 'TURN07'], dtype=object)

### H

Estaciones

In [96]:
df_unificado[df_unificado['LINEA'] == 'H']['ESTACION'].unique()

array(['CASEROS', 'CORRIENTES', 'HOSPITALES', 'HUMBERTO I', 'INCLAN',
       'ONCE', 'PATRICIOS', 'VENEZUELA', 'LAS HERAS', 'CORDOBA',
       'SANTA FE', 'FACULTAD DE DERECHO'], dtype=object)

Bocas

In [97]:
df_unificado[df_unificado['LINEA'] == 'H']['BOCA'].unique()

array(['H_CASEROS_NORTE', 'H_CASEROS_SUR', 'H_CORRIENTES_NORTE',
       'H_CORRIENTES_SUR', 'H_HOSPITALES_SUR', 'H_HOSPITALES_NORTE',
       'H_HPRIMO_SUR', 'H_HPRIMO_NORTE', 'H_INCLAN_NORTE', 'H_INCLAN_SUR',
       'H_ONCE_NORTE', 'H_ONCE_SUR', 'H_PATRICIOS', 'H_VENEZUELA_SUR',
       'H_VENEZUELA_NORTE', 'HPRIMO_SUR', 'VENEZUELA_NORTE',
       'CASEROS_NORTE', 'HOSPITALES_SUR', 'INCLAN_NORTE', 'ONCE_SUR',
       'HOSPITALES_NORTE', 'PATRICIOS', 'CASEROS_SUR', 'CORRIENTES_NORTE',
       'VENEZUELA_SUR', 'CORRIENTES_SUR', 'HPRIMO_NORTE', 'ONCE_NORTE',
       'INCLAN_SUR', 'H_LASHERAS', 'H_CORDOBA', 'H_SANTAFE_NORTE',
       'H_SANTAFE_SUR', 'SANTAFE_NORTE', 'SANTAFE_SUR', 'SANTAFE_ASC',
       'CORDOBA', 'LASHERAS', 'FDERECHO_SUR', 'FDERECHO_NORTE'],
      dtype=object)

In [98]:
df_unificado.loc[df_unificado['LINEA'] == 'H', 'BOCA'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'H', 'BOCA'].str.replace(r'^H_', '', regex=True)
)

In [99]:
df_unificado[df_unificado['LINEA'] == 'H']['BOCA'].unique()

array(['CASEROS_NORTE', 'CASEROS_SUR', 'CORRIENTES_NORTE',
       'CORRIENTES_SUR', 'HOSPITALES_SUR', 'HOSPITALES_NORTE',
       'HPRIMO_SUR', 'HPRIMO_NORTE', 'INCLAN_NORTE', 'INCLAN_SUR',
       'ONCE_NORTE', 'ONCE_SUR', 'PATRICIOS', 'VENEZUELA_SUR',
       'VENEZUELA_NORTE', 'LASHERAS', 'CORDOBA', 'SANTAFE_NORTE',
       'SANTAFE_SUR', 'SANTAFE_ASC', 'FDERECHO_SUR', 'FDERECHO_NORTE'],
      dtype=object)

Molinetes

In [100]:
df_unificado[df_unificado['LINEA'] == 'H']['MOLINETE'].unique()

array(['TURN01', 'TURN02', 'TURN04', 'TURN03', 'TURN05', 'TURN06'],
      dtype=object)

## Cuarta etapa: chequeamos duplicados, nulos y guardamos

Chequeamos por duplicados y borramos.

In [101]:
df_unificado.duplicated().sum()

np.int64(40839)

In [102]:
df_unificado.drop_duplicates(inplace=True)

In [103]:
df_unificado.duplicated().sum()

np.int64(0)

Guardamos en archivos diferenciados por año nuevamente

In [ ]:
output_dir = "../output/13-21"
os.makedirs(output_dir, exist_ok=True)

for anio, df_grupo in df_unificado.groupby(df_unificado['FECHA'].dt.year):
    ruta_archivo = os.path.join(output_dir, f"df{anio}.parquet")
    
    df_grupo.to_parquet(
        ruta_archivo, 
        index=False, 
        engine='pyarrow', 
        compression='snappy',
        coerce_timestamps='ms'
    )
    print(f"Año {anio} guardado exitosamente ({len(df_grupo):,} filas).")

Año 2013 guardado exitosamente (6,140,168 filas).
Año 2014 guardado exitosamente (10,857,244 filas).
Año 2015 guardado exitosamente (10,958,582 filas).
Año 2016 guardado exitosamente (11,542,322 filas).
Año 2017 guardado exitosamente (11,938,449 filas).
Año 2018 guardado exitosamente (12,058,191 filas).
Año 2019 guardado exitosamente (12,662,343 filas).
Año 2020 guardado exitosamente (5,760,127 filas).
Año 2021 guardado exitosamente (8,049,884 filas).


# Conjunto 2022-2026

Para este conjunto, vamos a reutilizar buena parte del código de la primera etapa. La principal diferencia estará en que primero vamos a hacer un split en la primer fila de cada csv (las columnas) para quitarles las comillas. Por otro lado, al tener múltiples archivos csv por cada año, se requiere hacer un recorrido recursivo de cada carpeta (a diferencia del conjunto anterior que era 1 archivo por cada año).

## Primera Etapa: estandarización y enriquecimiento

Necesitamos importar el módulo _"from\_path"_ de la librería _"charset\_normalizer"_.

In [105]:
import os

import pandas as pd
from charset_normalizer import from_path

Volvemos a definir directorios de entrada y de salida. Creamos la carpeta de salida, definimos columnas y definimos el _fallback_. Notese que no definimos la muestra de bytes para el encoding. El abordaje va a ser levemente distinto.

In [ ]:
base_path = "../dataset/2022-2026/"
output_folder = "../output/22-26/"
os.makedirs(output_folder, exist_ok=True)
columnas_finales = ['FECHA', 'DESDE', 'HASTA', 'HORA', 'LINEA', 'ESTACION', 'BOCA', 'MOLINETE', 'MOLINETE_ORIGINAL', 'PAX_TOTAL']

FALLBACK_ENCODING = 'latin-1'

La función "detectar_encoding" es levemente distinta a la del conjunto anterior. Vamos a escanear el documento completo, al ser más pequeño cada uno, podemos hacerlo con menos recursos y la lectura es más precisa.

In [107]:
def detectar_encoding(filepath):
    resultado = from_path(filepath).best()
    if resultado is not None:
        return resultado.encoding
    return None

Las siguientes funciones son idénticas al conjunto anterior. Se elimina la función _"leer\_csv\_con\_fallback"_ y ese proceso es reemplazado por otro en la función _"procesar\_archivo"_.

In [108]:

def detectar_separador(filepath, encoding):
    """Detecta el separador de un archivo CSV leyendo la primera línea. Verifica primero por ';' y por defecto usa ','."""
    with open(filepath, 'r', encoding=encoding, errors='replace') as f:
        header = f.readline()
    if ';' in header:
        return ';'
    return ','

def parseo_en_cascada(serie):
    # Opción 1: ISO 8601 (sin ambigüedades)
    df_split = pd.to_datetime(serie, format='%Y-%m-%d', errors='coerce')

    # Opción 2: Día primero (ej. 1/6/2022 o 01/06/2022)
    df = df_split.fillna(pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce'))

    # Opción 3: Mes primero (ej. 06/01/2013) para los que fallaron por día > 12 en mes
    df = df.fillna(pd.to_datetime(serie, format='%m/%d/%Y', errors='coerce'))

    # Opción 4: Fallback con dayfirst explícito para cualquier caso restante
    df = df.fillna(pd.to_datetime(serie, dayfirst=True, errors='coerce'))
    
    return df

La función _"procesar\_archivo"_ tiene una diferencia fundamental con la del conjunto anterior.

En un archivo CSV estándar, las comillas se usan para proteger campos individuales que contienen el delimitador dentro (ej. "Loria, Norte"). Sin embargo, en el conjunto de datos problemático, la línea entera venía entrecomillada:

"FECHA;DESDE;HASTA;LINEA;ESTACION;..."
"2022-01-01;06:00;06:15;A;ACOYTE;..."

Cuando Pandas intenta leer esto por defecto con sep=';', interpreta que todo el contenido entre la comilla inicial y final es un único valor de texto. Por lo tanto, no separa las columnas y colapsa toda la fila en una sola celda. Un dolor de cabeza.

Para solucionar este problema, la secuencia es la siguiente:

1) Neutralizar la interpretación de comillas (_quoting=3_ y _header=None_): esto crea una única columna y le quita el "rango" de encabezado a la primera fila.
2) Limpieza de caracteres y partición manual (_str.replace_ y _str.split_): buscamos todas las comillas y las borramos, luego separamos cada columna con el seprador correspondiente y expandimos las columnas (_.str.split(sep, expand=True)_)
3) Reconstrucción de los encabezados: _df_split.columns = df_split.iloc[0].str.strip().str.upper()_ vuelve a definir como columnas a esta primer fila que había quedado "cruda". Luego, con _df_split = df_split[1:].reset_index(drop=True)_ eliminamos la fila con las columnas que nos quedó, y reseteamos el índice para que vuelva a 0.


In [ ]:
def procesar_archivo(filepath, archivo):
    print(f"  Detectando encoding de: {archivo}")
    encoding = detectar_encoding(filepath)
    print(f"   Codec detectado: {encoding}")

    sep = detectar_separador(filepath, encoding)
    print(f"   Separador detectado: '{sep}'")

    print(f"   Procesando {archivo}")

    df_raw = pd.read_csv(filepath, encoding=encoding, header=None, quoting=3) # header=None para que no tome la primera fila como nombres de columna, quoting=3 para ignorar comillas
                
    df_split = (df_raw[0].str.replace('"', '').str.split(sep, expand=True)) # expand=True para crear columnas separadas
    
    df_split.columns = df_split.iloc[0].str.strip().str.upper() # df_split.iloc[0] toma la primera fila como nombres de columna (indice 0)
    df_split = df_split[1:].reset_index(drop=True) # df_split[1:] descarta la primera fila (nombres de columna) y reset_index(drop=True) reinicia el índice del DataFrame

    # --- Normalizar columnas ---
    df_split.rename(columns={'TOTAL': 'PAX_TOTAL'}, inplace=True)

    # --- Transformaciones de texto ---
    
    df_split['LINEA'] = df_split['LINEA'].str.strip().str[-1]
    df_split['ESTACION'] = df_split['ESTACION'].str.strip().str.upper()

    df_split['MOLINETE_ORIGINAL'] = df_split['MOLINETE']

    partes = df_split['MOLINETE'].str.extract(r'^(?P<PREFIJO>[^_]+)_(?P<BOCA>.+)_(?P<MOLINETE>[^_]+)$')

    df_split['BOCA'] = partes['BOCA'].str.strip().str.upper()
    df_split['MOLINETE'] = partes['MOLINETE'].str.strip().str.upper()

    # --- Formatos Datetime --- 

    desde_dt = pd.to_datetime(df_split['DESDE'], format='mixed', errors='coerce')
    hasta_dt = pd.to_datetime(df_split['HASTA'], format='mixed', errors='coerce')

    df_split['HORA'] = desde_dt.dt.hour.astype('Int64') # Int64 maneja valores nulos si falla el parseo
    df_split['DESDE'] = desde_dt.dt.strftime('%H:%M')
    df_split['HASTA'] = hasta_dt.dt.strftime('%H:%M')

    df_split['FECHA'] = parseo_en_cascada(df_split['FECHA'])

    df_split['FECHA'] = df_split['FECHA'].astype('datetime64[ms]')

    df_split['PAX_TOTAL'] = pd.to_numeric(df_split['PAX_TOTAL'], errors='coerce').astype('Int64')

    # --- Seleccionar y guardar ---
    df = df_split[columnas_finales]
    return df

Loop principal de ejecución y guardado:

In [110]:
# --- Loop principal ---

for folder_name in sorted(os.listdir(base_path)):
    folderpath = os.path.join(base_path, folder_name)

    if not os.path.isdir(folderpath):
        continue

    print(f"Leyendo carpeta: {folder_name}")
    dataframes = []

    for archivo in os.listdir(folderpath):
        if archivo.endswith('.csv'):
            filepath = os.path.join(folderpath, archivo)
            df = procesar_archivo(filepath, archivo)
            dataframes.append(df)
    if dataframes:
        df_concat = pd.concat(dataframes, ignore_index=True)
        numero = ''.join(filter(str.isdigit, archivo))[:4]
        nombre_salida = f"df{numero}.parquet"
        ruta_salida = os.path.join(output_folder, nombre_salida)
        df_concat.to_parquet(
        ruta_salida,
        index=False,
        engine='pyarrow',
        compression='snappy',
        coerce_timestamps='ms'
        )
        print(f"                Guardado en ------------> {ruta_salida}")

Leyendo carpeta: molinetes-2022
  Detectando encoding de: 202201_PAX15min-ABC.csv
   Codec detectado: cp1250
   Separador detectado: ';'
   Procesando 202201_PAX15min-ABC.csv
  Detectando encoding de: 202201_PAX15min-DEH.csv
   Codec detectado: cp1250
   Separador detectado: ';'
   Procesando 202201_PAX15min-DEH.csv
  Detectando encoding de: 202202_PAX15min-ABC.csv
   Codec detectado: cp1250
   Separador detectado: ';'
   Procesando 202202_PAX15min-ABC.csv
  Detectando encoding de: 202202_PAX15min-DEH.csv
   Codec detectado: cp1250
   Separador detectado: ';'
   Procesando 202202_PAX15min-DEH.csv
  Detectando encoding de: 202203_PAX15min-ABC.csv
   Codec detectado: cp1250
   Separador detectado: ';'
   Procesando 202203_PAX15min-ABC.csv
  Detectando encoding de: 202203_PAX15min-DEH.csv
   Codec detectado: cp1250
   Separador detectado: ';'
   Procesando 202203_PAX15min-DEH.csv
  Detectando encoding de: 202204_PAX15min-ABC.csv
   Codec detectado: cp1250
   Separador detectado: ';'
   Pr

## Segunda Etapa: detección de nulos

#### Buscamos filas completamente vacías: si existen las eliminamos, y volvemos a guardar los archivos.

In [ ]:
folderpath = "../output/22-26/"
output_folder = "./output/22-26" # usamos la misma ruta para directamente sobreescribir los archivos procesados
sum = 0
os.makedirs(output_folder, exist_ok=True)

for archivo in sorted(os.listdir(folderpath)):
    if archivo.endswith('.parquet'):
        filepath = os.path.join(folderpath, archivo)
        output_filepath = os.path.join(output_folder, archivo)
        
        print(f"\nProcesando: {archivo}")
        
        # 1. Lectura del archivo individual
        df = pd.read_parquet(filepath, engine='pyarrow')
        
        # 2. Verificación y limpieza de filas completamente vacías
        filas_nulas = df.isna().all(axis=1).sum()
        if filas_nulas > 0:
            print(f"  - Se detectaron {filas_nulas} filas completamente nulas. Eliminando...")
            df.dropna(inplace=True, how='all', axis=0)
            print("  - Filas eliminadas con éxito.")
            sum += filas_nulas
        else:
            print("  - No se detectaron filas completamente nulas.")
            
        # 3. Guardado en la carpeta de destino con compresión snappy
        df.to_parquet(
            output_filepath,
            index=False,
            engine='pyarrow',
            compression='snappy',
            coerce_timestamps='ms'
        )
        print(f"  - Guardado en: {output_filepath}")
print(f"\nTotal de filas completamente nulas eliminadas en todos los archivos: {sum}")


Procesando: df2022.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/22-26\df2022.parquet

Procesando: df2023.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/22-26\df2023.parquet

Procesando: df2024.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/22-26\df2024.parquet

Procesando: df2025.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/22-26\df2025.parquet

Procesando: df2026.parquet
  - No se detectaron filas completamente nulas.
  - Guardado en: ./output/22-26\df2026.parquet

Total de filas completamente nulas eliminadas en todos los archivos: 0


### Chequeamos si hay filas con algún valor nulo

In [112]:
folderpath = "./output/22-26"
lista_nulos = []

for archivo in sorted(os.listdir(folderpath)):
    if archivo.endswith('.parquet'):
        filepath = os.path.join(folderpath, archivo)
        
        print(f"\nProcesando: {archivo}")
        
        # 1. Lectura del archivo individual
        df = pd.read_parquet(filepath, engine='pyarrow')
        
        # 2. Verificación y limpieza de filas con valores nulos
        filas_nulas = df.isna().any(axis=1).sum()
        if filas_nulas > 0:
            print(f"  - Se detectaron {filas_nulas} filas con valores nulos.")
            lista_nulos.append(archivo)
        else:
            print("  - No se detectaron filas con valores nulos.")
print(f"\nLos archivos con filas nulas son: {lista_nulos}")


Procesando: df2022.parquet
  - No se detectaron filas con valores nulos.

Procesando: df2023.parquet
  - No se detectaron filas con valores nulos.

Procesando: df2024.parquet
  - Se detectaron 158033 filas con valores nulos.

Procesando: df2025.parquet
  - Se detectaron 169009 filas con valores nulos.

Procesando: df2026.parquet
  - Se detectaron 249083 filas con valores nulos.

Los archivos con filas nulas son: ['df2024.parquet', 'df2025.parquet', 'df2026.parquet']


### Revisamos uno por uno los archivos para ver qué valores faltan

#### 2024

In [ ]:
df = pd.read_parquet("../output/22-26/df2024.parquet", engine='pyarrow')
df.isna().sum()

FECHA                     0
DESDE                     0
HASTA                     0
HORA                      0
LINEA                     0
ESTACION                  0
BOCA                 158033
MOLINETE             158033
MOLINETE_ORIGINAL         0
PAX_TOTAL                 0
dtype: int64

In [114]:
df[df['BOCA'].isna()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
10954074,2024-12-02,05:15,05:30,5,A,ACOYTE,None,None,null,1
10954075,2024-12-02,05:30,05:45,5,A,ACOYTE,None,None,null,1
10954076,2024-12-02,05:45,06:00,5,A,ACOYTE,None,None,null,1
10954077,2024-12-02,06:15,06:30,6,A,ACOYTE,None,None,null,1
10954078,2024-12-02,06:30,06:45,6,A,ACOYTE,None,None,null,4
...,...,...,...,...,...,...,...,...,...,...
11440435,2024-12-31,19:00,19:15,19,H,VENEZUELA,None,None,null,2
11440436,2024-12-31,19:15,19:30,19,H,VENEZUELA,None,None,null,3
11440437,2024-12-31,19:30,19:45,19,H,VENEZUELA,None,None,null,3
11440438,2024-12-31,19:45,20:00,19,H,VENEZUELA,None,None,null,4


Vemos que el dataset no trae la información de molinetes. Reemplazamos por _SIN ESPECIFICAR_.

In [115]:
df['BOCA'].fillna('SIN ESPECIFICAR', inplace=True)
df['MOLINETE'].fillna('SIN ESPECIFICAR', inplace=True)

Eliminamos duplicados

In [116]:
df.duplicated().sum()

np.int64(3)

In [117]:
df.drop_duplicates(inplace=True)

Reemplazamos archivo:

In [ ]:
df.to_parquet("../output/22-26/df2024.parquet", engine='pyarrow', index=False, compression='snappy', coerce_timestamps='ms')

In [ ]:
df = pd.read_parquet("../output/22-26/df2024.parquet", engine='pyarrow')
df.isna().sum()

FECHA                0
DESDE                0
HASTA                0
HORA                 0
LINEA                0
ESTACION             0
BOCA                 0
MOLINETE             0
MOLINETE_ORIGINAL    0
PAX_TOTAL            0
dtype: int64

#### 2025

In [120]:
df = pd.read_parquet("./output/22-26/df2025.parquet", engine='pyarrow')
df.isna().sum()

FECHA                     0
DESDE                     0
HASTA                     0
HORA                      0
LINEA                     0
ESTACION                  0
BOCA                 169009
MOLINETE             169009
MOLINETE_ORIGINAL         0
PAX_TOTAL                 0
dtype: int64

In [121]:
df[df['BOCA'].isna()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
434088,2025-01-01,08:30,08:45,8,A,ACOYTE,None,None,NULL,2
434089,2025-01-01,08:45,09:00,8,A,ACOYTE,None,None,NULL,2
434090,2025-01-01,09:30,09:45,9,A,ACOYTE,None,None,NULL,1
434091,2025-01-01,09:45,10:00,9,A,ACOYTE,None,None,NULL,3
434092,2025-01-01,10:00,10:15,10,A,ACOYTE,None,None,NULL,1
...,...,...,...,...,...,...,...,...,...,...
927644,2025-01-31,22:00,22:15,22,H,VENEZUELA,None,None,NULL,4
927645,2025-01-31,22:15,22:30,22,H,VENEZUELA,None,None,NULL,3
927646,2025-01-31,22:30,22:45,22,H,VENEZUELA,None,None,NULL,1
927647,2025-01-31,22:45,23:00,22,H,VENEZUELA,None,None,NULL,2


Caso similar al anterior: el campo molinete vino vacío del crudo. Reemplazamos por _SIN ESPECIFICAR_.

In [122]:
df['BOCA'].fillna('SIN ESPECIFICAR', inplace=True)
df['MOLINETE'].fillna('SIN ESPECIFICAR', inplace=True)

Descartamos fila que dice "PRUEBA":

In [123]:
df = df[(df['ESTACION'] != "PRUEBA")]

Chequeamos y eliminamos duplicados

In [124]:
df.duplicated().sum()

np.int64(64143)

In [125]:
df.drop_duplicates(inplace=True)

Reemplazamos archivo:

In [126]:
df.to_parquet("./output/22-26/df2025.parquet", engine='pyarrow', index=False, compression='snappy', coerce_timestamps='ms')

In [127]:
df = pd.read_parquet("./output/22-26/df2025.parquet", engine='pyarrow')
df.isna().sum()

FECHA                0
DESDE                0
HASTA                0
HORA                 0
LINEA                0
ESTACION             0
BOCA                 0
MOLINETE             0
MOLINETE_ORIGINAL    0
PAX_TOTAL            0
dtype: int64

#### 2026

In [128]:
df = pd.read_parquet("./output/22-26/df2026.parquet", engine='pyarrow')
df.isna().sum()

FECHA                     0
DESDE                     0
HASTA                249082
HORA                      0
LINEA                     0
ESTACION                  0
BOCA                      1
MOLINETE                  1
MOLINETE_ORIGINAL         0
PAX_TOTAL                 0
dtype: int64

In [129]:
df[df['HASTA'].isna()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
6836783,2026-06-01,04:15,None,4,D,CALLAO.D,CALLAOD_S,TURN01,LineaD_CallaoD_S_Turn01,1
6836784,2026-06-01,04:15,None,4,D,CALLAO.D,CALLAOD_E,TURN01,LineaD_CallaoD_E_Turn01,1
6836785,2026-06-01,04:15,None,4,D,CONGRESO DE TUCUMAN,CONGRESOTUC_O,TURN07,LineaD_CongresoTuc_O_Turn07,1
6836786,2026-06-01,04:15,None,4,D,OLLEROS,OLLEROS_OESTE,TURN01,LineaD_Olleros_Oeste_Turn01,1
6836787,2026-06-01,04:15,None,4,D,CONGRESO DE TUCUMAN,CONGRESOTUC_O,TURN01,LineaD_CongresoTuc_O_Turn01,1
...,...,...,...,...,...,...,...,...,...,...
7085860,2026-06-30,23:15,None,23,H,HUMBERTO I,HPRIMO_SUR,TURN01,LineaH_HPrimo_Sur_Turn01,2
7085861,2026-06-30,23:15,None,23,H,CORDOBA,CORDOBA,TURN04,LineaH_Cordoba_Turn04,3
7085862,2026-06-30,23:15,None,23,H,FACULTAD DE DERECHO,FDERECHO_SUR,TURN05,LineaH_FDerecho_Sur_Turn05,1
7085863,2026-06-30,23:15,None,23,H,LAS HERAS,LASHERAS,TURN04,LineaH_LasHeras_Turn04,3


Sumamos 14 minutos al campo HASTA de los valores faltantes:

In [130]:
calculado = (pd.to_datetime(df['DESDE'], format='%H:%M') + pd.Timedelta(minutes=14)).dt.strftime('%H:%M')

df['HASTA'] = df['HASTA'].fillna(calculado)

In [131]:
df[df['BOCA'].isna()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
7158882,2026-06-16,12:00,12:15,12,a,PRUEBA,None,None,Prueba,2


Descartamos fila que dice "PRUEBA":

In [132]:
df = df[(df['ESTACION'] != "PRUEBA")]

Descartamos otra fila de prueba:

In [133]:
df[(df['MOLINETE_ORIGINAL']=='LineaH_Validador_Central_Turn01')]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
1676186,2026-02-25,10:45,11:00,10,H,#N/D,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,1


In [134]:
df = df[(df['MOLINETE_ORIGINAL'] != "LineaH_Validador_Central_Turn01")]

Eliminamos duplicados

In [135]:
df.duplicated().sum()

np.int64(73814)

In [136]:
df.drop_duplicates(inplace=True)

Reemplazamos archivo:

In [137]:
df.to_parquet("./output/22-26/df2026.parquet", engine='pyarrow', index=False, compression='snappy', coerce_timestamps='ms')

In [138]:
df = pd.read_parquet("./output/22-26/df2026.parquet", engine='pyarrow')
df.isna().sum()

FECHA                0
DESDE                0
HASTA                0
HORA                 0
LINEA                0
ESTACION             0
BOCA                 0
MOLINETE             0
MOLINETE_ORIGINAL    0
PAX_TOTAL            0
dtype: int64

## Tercera etapa: tratamiento de nulos y normalización de estaciones, bocas, y molinetes

Acá vamos a revisar en profundidad todo el conjunto 2022-2026. Vamos linea por linea. Buscamos inconsistencias en los nombres, filas incorrectas, caracteres extraños, estaciones incorrectas, filas de prueba, etc. Este paso es crítico para tener un dataset bien prolijo.

También rellenamos valores nulos según el tipo de dato. Se detallará a continuación cada caso.

Este proceso es casi idéntico al conjunto anterior, simplemente salvaguardando las particularidades de este conjunto, y con la adición del _premetro_. 

### Concatenamos en un solo df en memoria para diagnóstico general

Concatenar en un sólo _dataframe_ nos permite trabajar con mayor comodidad.

In [ ]:
df_unificado = pd.read_parquet("../output/22-26", engine='pyarrow')
df_unificado['FECHA'].min(), df_unificado['FECHA'].max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2026-06-30 00:00:00'))

In [140]:
df_unificado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55842927 entries, 0 to 55842926
Data columns (total 10 columns):
 #   Column             Dtype         
---  ------             -----         
 0   FECHA              datetime64[ms]
 1   DESDE              object        
 2   HASTA              object        
 3   HORA               Int64         
 4   LINEA              object        
 5   ESTACION           object        
 6   BOCA               object        
 7   MOLINETE           object        
 8   MOLINETE_ORIGINAL  object        
 9   PAX_TOTAL          Int64         
dtypes: Int64(2), datetime64[ms](1), object(7)
memory usage: 4.3+ GB


### A

In [141]:
df_unificado[df_unificado['LINEA'] == "A"]['ESTACION'].unique()

array(['CARABOBO', 'LIMA', 'PERU', 'ACOYTE', 'CASTRO BARROS', 'FLORES',
       'LORIA', 'PLAZA DE MAYO', 'CONGRESO', 'PUAN', 'RIO DE JANEIRO',
       'SAENZ PEŃA', 'PLAZA MISERERE', 'SAN PEDRITO', 'PIEDRAS', 'PASCO',
       'ALBERTI', 'PRIMERA JUNTA', 'SAENZ PE±A', 'SAENZ PEÑA'],
      dtype=object)

In [142]:
df_unificado.loc[
    (df_unificado['LINEA'] == 'A') & 
    (df_unificado['ESTACION'].str.contains('SAENZ', case=False, na=False)),
    'ESTACION'
] = 'SAENZ PEÑA'

In [143]:
df_unificado[df_unificado['LINEA'] == "A"]['ESTACION'].unique()

array(['CARABOBO', 'LIMA', 'PERU', 'ACOYTE', 'CASTRO BARROS', 'FLORES',
       'LORIA', 'PLAZA DE MAYO', 'CONGRESO', 'PUAN', 'RIO DE JANEIRO',
       'SAENZ PEÑA', 'PLAZA MISERERE', 'SAN PEDRITO', 'PIEDRAS', 'PASCO',
       'ALBERTI', 'PRIMERA JUNTA'], dtype=object)

In [144]:
df_unificado[df_unificado['LINEA'] == "A"]['BOCA'].unique()

array(['CARABOBO_E', 'LIMA_S', 'PERU_S', 'ACOYTE_S', 'CBARROS_S',
       'FLORES_OESTE', 'LORIA_S', 'PMAYO_E', 'ACOYTE_N', 'CONGRESO_N',
       'PUAN_E', 'RJANEIRO_N', 'SNZPENA_S', 'CARABOBO_O', 'FLORES_ESTE',
       'LORIA_N', 'MISERERE_Q_HALL', 'CBARROS_N', 'MISERERE_S',
       'SANPEDRITO_ESTE', 'CONGRESO_S', 'PIEDRAS_S', 'PASCO', 'ALBERTI',
       'SNZPENA_N', 'PUAN_O', 'SANPEDRITO_OESTE', 'RJANEIRO_S', 'PERU_N',
       'LIMA_N', 'PJUNTA_S', 'PMAYO_O', 'PJUNTA_N', 'PIEDRAS_N',
       'MISERERE_Q_NE', 'SIN ESPECIFICAR'], dtype=object)

In [145]:
df_unificado[df_unificado['LINEA'] == "A"]['MOLINETE'].unique()

array(['TURN02', 'TURN01', 'TURN03', 'TURN04', 'TURN05', 'TURN06',
       'ASC01', 'DISCAP04', 'SIN ESPECIFICAR', 'TURN01-EMV+QR',
       'TURN05-EMV', 'TURN04-EMV'], dtype=object)

### B

In [146]:
df_unificado[df_unificado['LINEA'] == "B"]['ESTACION'].unique()

array(['ECHEVERRIA', 'LOS INCAS', 'URUGUAY', 'FEDERICO LACROZE',
       'MALABIA', 'CALLAO.B', 'DORREGO', 'ANGEL GALLARDO',
       'CARLOS GARDEL', 'ROSAS', 'LEANDRO N. ALEM', 'MEDRANO', 'PASTEUR',
       'CARLOS PELLEGRINI', 'PUEYRREDON', 'FLORIDA', 'TRONADOR',
       'PUEYRREDON.B', 'LORIA'], dtype=object)

In [147]:
df_unificado.loc[df_unificado['LINEA'] == 'B', 'ESTACION'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'B', 'ESTACION'].replace({
        "CALLAO.B": "CALLAO",
        "PUEYRREDON.B": "PUEYRREDON"})
)

La estación "Loria" no existe en la línea B.

In [148]:
df_unificado = df_unificado[~((df_unificado['LINEA'] == "B") & (df_unificado['ESTACION'] == 'LORIA'))]

In [149]:
df_unificado[df_unificado['LINEA'] == "B"]['ESTACION'].unique()

array(['ECHEVERRIA', 'LOS INCAS', 'URUGUAY', 'FEDERICO LACROZE',
       'MALABIA', 'CALLAO', 'DORREGO', 'ANGEL GALLARDO', 'CARLOS GARDEL',
       'ROSAS', 'LEANDRO N. ALEM', 'MEDRANO', 'PASTEUR',
       'CARLOS PELLEGRINI', 'PUEYRREDON', 'FLORIDA', 'TRONADOR'],
      dtype=object)

In [150]:
df_unificado[df_unificado['LINEA'] == "B"]['BOCA'].unique()

array(['ECHEVERRIA_OESTE', 'LOSINCAS', 'URUGUAY_S', 'LACROZE_O',
       'MALABIA_N', 'CALLAOB_S', 'DORREGO_N', 'GALLARDO_N', 'GARDEL_S',
       'JMROSAS_ESTE', 'ALEM_N', 'GARDEL_N', 'JMROSAS_OESTE', 'MALABIA_S',
       'MEDRANO_S', 'PASTEUR_S', 'PELLEGRINI_E', 'PUEYR_N', 'CALLAOB_N',
       'ALEM_S', 'MEDRANO_N', 'PUEYR_S', 'ECHEVERRIA_ESTE', 'GALLARDO_S',
       'LACROZE_S', 'FLORIDA_O', 'LACROZE_E', 'DORREGO_S', 'TRONADOR',
       'URUGUAY_N', 'PASTEUR_N', 'FLORIDA_E', 'PELLEGRINI_O',
       'SIN ESPECIFICAR', 'GRADEL_N'], dtype=object)

In [151]:
df_unificado[df_unificado['LINEA'] == "B"]['MOLINETE'].unique()

array(['TURN02', 'TURN03', 'TURN04', 'TURN05', 'TURN06', 'TURN01',
       'TURN09', 'TURN07', 'TURN08', 'ASC01', 'SIN ESPECIFICAR',
       'TURN06-EMV+QR', 'TURN03-EMV'], dtype=object)

### C

In [152]:
df_unificado[df_unificado['LINEA'] == "C"]['ESTACION'].unique()

array(['DIAGONAL NORTE', 'INDEPENDENCIA', 'CONSTITUCION', 'LAVALLE',
       'RETIRO', 'AVENIDA DE MAYO', 'GENERAL SAN MARTIN', 'SAN JUAN',
       'MARIANO MORENO', 'INDEPENDENCIA.C', 'RETIRO.C', 'LORIA'],
      dtype=object)

In [153]:
df_unificado.loc[df_unificado['LINEA'] == 'C', 'ESTACION'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'C', 'ESTACION'].replace({
        "RETIRO.C": "RETIRO",
        "INDEPENDENCIA.C": "INDEPENDENCIA"})
)

In [154]:
df_unificado[df_unificado['LINEA'] == "C"]['ESTACION'].unique()

array(['DIAGONAL NORTE', 'INDEPENDENCIA', 'CONSTITUCION', 'LAVALLE',
       'RETIRO', 'AVENIDA DE MAYO', 'GENERAL SAN MARTIN', 'SAN JUAN',
       'MARIANO MORENO', 'LORIA'], dtype=object)

In [155]:
df_unificado[(df_unificado['LINEA'] == "C") & (df_unificado['ESTACION'] == 'LORIA')]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
36134971,2025-01-22,22:45,23:00,22,C,LORIA,SIN ESPECIFICAR,SIN ESPECIFICAR,NULL,2
36134972,2025-01-22,23:00,23:15,23,C,LORIA,SIN ESPECIFICAR,SIN ESPECIFICAR,NULL,1


Se detecta una inconsistencia, la estación "LORIA" no existe en la linea C, procedemos a eliminar.

In [156]:
df_unificado = df_unificado[~((df_unificado['LINEA'] == "C") & (df_unificado['ESTACION'] == 'LORIA'))]

In [157]:
df_unificado[df_unificado['LINEA'] == "C"]['ESTACION'].unique()

array(['DIAGONAL NORTE', 'INDEPENDENCIA', 'CONSTITUCION', 'LAVALLE',
       'RETIRO', 'AVENIDA DE MAYO', 'GENERAL SAN MARTIN', 'SAN JUAN',
       'MARIANO MORENO'], dtype=object)

In [158]:
df_unificado[df_unificado['LINEA'] == "C"]['BOCA'].unique()

array(['DNORTE_S', 'INDEPEN', 'CONSTITUCION_PLAZA', 'LAVALLE_S', 'RETIRO',
       'AVMAYO_S', 'CONSTITUCION', 'SANMARTIN_N', 'SANJUAN',
       'SANMARTIN_S', 'MORENO_S', 'DNORTE_N', 'MORENO_N', 'AVMAYO_N',
       'LAVALLE_N', 'SIN ESPECIFICAR'], dtype=object)

In [159]:
df_unificado[df_unificado['LINEA'] == "C"]['MOLINETE'].unique()

array(['TURN01', 'TURN04', 'TURN08', 'TURN09', 'TURN10', 'TURN11',
       'TURN02', 'TURN07', 'TURN03', 'TURN12', 'TURN13', 'TURN05',
       'TURN17', 'TURN06', 'TURN16', 'TURN18', 'TURN14', 'TURN15',
       'TURN24', 'TURN23', 'TURN25', 'TURN26', 'TURN21', 'TURN20',
       'TURN19', 'TURN22', 'TURN', 'TURN29', 'TURN27', 'ASC01', 'TURN28',
       'SIN ESPECIFICAR', 'TURNEMV+QR', 'TURN00-EMV+QR', 'TURN05-EMV+QR',
       'TURN01-EMV+QR', 'TURN05-EMV'], dtype=object)

### D

In [160]:
df_unificado[df_unificado['LINEA'] == "D"]['ESTACION'].unique()

array(['AGÜERO', 'CALLAO', 'JOSE HERNANDEZ', 'JURAMENTO', '9 DE JULIO',
       'CONGRESO DE TUCUMAN', 'PALERMO', 'MINISTRO CARRANZA',
       'FACULTAD DE MEDICINA', 'BULNES', 'SCALABRINI ORTIZ', 'TRIBUNALES',
       'OLLEROS', 'CATEDRAL', 'PLAZA ITALIA', 'PUEYRREDON.D', 'CALLAO.D',
       'LORIA'], dtype=object)

Detectemos mismo error que en la linea C, la estación "LORIA" no existe en la linea D.

In [161]:
df_unificado.loc[df_unificado['LINEA'] == 'D', 'ESTACION'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'D', 'ESTACION'].replace({
        "PUEYRREDON.D": "PUEYRREDON",
        "CALLAO.D": "CALLAO"})
)

In [162]:
df_unificado = df_unificado[~((df_unificado['LINEA'] == "D") & (df_unificado['ESTACION'] == 'LORIA'))]

In [163]:
df_unificado[df_unificado['LINEA'] == "D"]['ESTACION'].unique()

array(['AGÜERO', 'CALLAO', 'JOSE HERNANDEZ', 'JURAMENTO', '9 DE JULIO',
       'CONGRESO DE TUCUMAN', 'PALERMO', 'MINISTRO CARRANZA',
       'FACULTAD DE MEDICINA', 'BULNES', 'SCALABRINI ORTIZ', 'TRIBUNALES',
       'OLLEROS', 'CATEDRAL', 'PLAZA ITALIA', 'PUEYRREDON'], dtype=object)

In [164]:
df_unificado[df_unificado['LINEA'] == "D"]['MOLINETE'].unique()

array(['TURN01', 'TURN02', 'TURN03', 'TURN05', 'TURN06', 'TURN04',
       'TURN08', 'ASC01', 'TURN07', 'TURN0', 'SIN ESPECIFICAR', 'TURN10',
       'TURN09', 'TURN11', 'TURN6-EMV+QR', 'TURN7-EMV+QR', 'TURN5-EMV+QR',
       'TURN04-EMV', 'TURN03-EMV', 'TURN7', 'TURN5', 'TURN6'],
      dtype=object)

### E

In [165]:
df_unificado[df_unificado['LINEA'] == "E"]['ESTACION'].unique()

array(['INDEPENDENCIA.H', 'BOLIVAR', 'SAN JOSE', 'CATALINAS',
       'ENTRE RIOS', 'CORREO CENTRAL', 'EMILIO MITRE', 'RETIRO E',
       'JUJUY', 'BOEDO', 'PZA. DE LOS VIRREYES', 'AVENIDA LA PLATA',
       'GENERAL BELGRANO', 'URQUIZA', 'PICHINCHA', 'JOSE MARIA MORENO',
       'VARELA', 'MEDALLA MILAGROSA', 'INDEPENDENCIA.E', 'RETIRO.E',
       'LORIA'], dtype=object)

In [166]:
df_unificado.loc[df_unificado['LINEA'] == 'E', 'ESTACION'] = (
    df_unificado.loc[df_unificado['LINEA'] == 'E', 'ESTACION'].replace({
        "INDEPENDENCIA.H": "INDEPENDENCIA",
        "INDEPENDENCIA.E": "INDEPENDENCIA",
        "RETIRO.E": "RETIRO E"}
        )
)

In [167]:
df_unificado = df_unificado[~((df_unificado['LINEA'] == "E") & (df_unificado['ESTACION'] == 'LORIA'))]

In [168]:
df_unificado[df_unificado['LINEA'] == "E"]['ESTACION'].unique()

array(['INDEPENDENCIA', 'BOLIVAR', 'SAN JOSE', 'CATALINAS', 'ENTRE RIOS',
       'CORREO CENTRAL', 'EMILIO MITRE', 'RETIRO E', 'JUJUY', 'BOEDO',
       'PZA. DE LOS VIRREYES', 'AVENIDA LA PLATA', 'GENERAL BELGRANO',
       'URQUIZA', 'PICHINCHA', 'JOSE MARIA MORENO', 'VARELA',
       'MEDALLA MILAGROSA'], dtype=object)

In [169]:
df_unificado[df_unificado['LINEA'] == "E"]['BOCA'].unique()

array(['INDEPEN', 'BOLIVAR_N', 'SANJOSE', 'CATALINAS_N', 'ERIOS',
       'CCENTRAL_S', 'EMITRE', 'RETIROE_N', 'JUJUY', 'RETIROE_S', 'BOEDO',
       'VIRREYES_E', 'LAPLATA', 'BELGRANO', 'CATALINAS_S', 'URQUIZA',
       'CCENTRAL_N', 'VIRREYES', 'PICHIN', 'MORENO', 'VARELA', 'MEDALLA',
       'BOLIVAR_S', 'SIN ESPECIFICAR'], dtype=object)

In [170]:
df_unificado[df_unificado['LINEA'] == "E"]['MOLINETE'].unique()

array(['TURN02', 'TURN04', 'TURN01', 'TURN06', 'TURN05', 'TURN03',
       'TURN07', 'SIN ESPECIFICAR', 'TURN00-EMV+QR', 'TURN11-EMV+QR',
       'TURN03-EMV', 'TURN04-EMV', 'TURN11'], dtype=object)

### H

In [171]:
df_unificado[df_unificado['LINEA'] == "H"]['ESTACION'].unique()

array(['HUMBERTO I', 'ONCE', 'CORRIENTES', 'HOSPITALES',
       'FACULTAD DE DERECHO', 'CASEROS', 'SANTA FE', 'VENEZUELA',
       'PATRICIOS', 'CORDOBA', 'LAS HERAS', 'INCLAN', 'LORIA', 'NULL',
       '#N/D'], dtype=object)

In [172]:
df_unificado = df_unificado[~((df_unificado['LINEA'] == "H") & (df_unificado['ESTACION'] == 'LORIA'))]

In [173]:
df_unificado[((df_unificado['LINEA'] == "H") & (df_unificado['ESTACION'] == '#N/D'))]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
45251520,2025-09-01,08:00,08:15,8,H,#N/D,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,2
46348353,2025-10-07,16:15,16:30,16,H,#N/D,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,2
46362270,2025-10-09,10:15,10:30,10,H,#N/D,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,1
46464695,2025-10-22,10:30,10:45,10,H,#N/D,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,1
46468179,2025-10-22,17:30,17:45,17,H,#N/D,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,1
48303147,2025-12-01,10:15,10:30,10,H,#N/D,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,1


Eliminamos filas invalidas:

In [174]:
df_unificado = df_unificado[~((df_unificado['LINEA'] == "H") & (df_unificado['ESTACION'] == '#N/D'))]

In [175]:
df_unificado[df_unificado['LINEA'] == "H"]['ESTACION'].unique()

array(['HUMBERTO I', 'ONCE', 'CORRIENTES', 'HOSPITALES',
       'FACULTAD DE DERECHO', 'CASEROS', 'SANTA FE', 'VENEZUELA',
       'PATRICIOS', 'CORDOBA', 'LAS HERAS', 'INCLAN', 'NULL'],
      dtype=object)

In [176]:
df_unificado[((df_unificado['LINEA'] == "H") & (df_unificado['ESTACION'] == 'NULL'))]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
42894636,2025-07-30,11:30,11:45,11,H,NULL,VALIDADOR_CENTRAL,TURN01,LineaH_Validador_Central_Turn01,1


In [177]:
df_unificado = df_unificado[~((df_unificado['LINEA'] == "H") & (df_unificado['ESTACION'] == 'NULL'))]

In [178]:
df_unificado[df_unificado['LINEA'] == "H"]['ESTACION'].unique()

array(['HUMBERTO I', 'ONCE', 'CORRIENTES', 'HOSPITALES',
       'FACULTAD DE DERECHO', 'CASEROS', 'SANTA FE', 'VENEZUELA',
       'PATRICIOS', 'CORDOBA', 'LAS HERAS', 'INCLAN'], dtype=object)

In [179]:
df_unificado[df_unificado['LINEA'] == "H"]['BOCA'].unique()

array(['HPRIMO_SUR', 'ONCE_NORTE', 'ONCE_SUR', 'CORRIENTES_NORTE',
       'HOSPITALES_SUR', 'FDERECHO_SUR', 'HOSPITALES_NORTE',
       'CASEROS_SUR', 'CORRIENTES_SUR', 'SANTAFE_NORTE', 'SANTAFE_SUR',
       'VENEZUELA_SUR', 'PATRICIOS', 'CORDOBA', 'LASHERAS',
       'INCLAN_NORTE', 'VENEZUELA_NORTE', 'HPRIMO_NORTE', 'CASEROS_NORTE',
       'FDERECHO_NORTE', 'INCLAN_SUR', 'SANTAFE_ASC', 'SIN ESPECIFICAR'],
      dtype=object)

In [180]:
df_unificado[df_unificado['LINEA'] == "H"]['MOLINETE'].unique()

array(['TURN01', 'TURN02', 'TURN06', 'TURN03', 'TURN04', 'TURN05',
       'SIN ESPECIFICAR'], dtype=object)

### PreMetro

In [181]:
df_unificado[df_unificado['LINEA'] == "M"]['ESTACION'].unique()

array(['SAGUIER', 'COCHEPM'], dtype=object)

In [182]:
df_unificado[df_unificado['LINEA'] == "M"]['BOCA'].unique()

array(['SAGUIER_N', 'COCHE11', 'COCHE1', 'SAGUIER_O', 'COCHE13',
       'SAGUIER_S', 'COCHE17', 'COCHE2', 'COCHE12', 'COCHE5', 'COCHE21',
       'COCHE14', 'COCHE7', 'COCHE9', 'COCHE10'], dtype=object)

In [183]:
df_unificado[df_unificado['LINEA'] == "M"]['MOLINETE'].unique()

array(['TURN02', 'TURN03', 'VAL2', 'VAL1', 'TURN04', 'TURN01', 'VAL4',
       'VAL3', 'TURN01-EMV'], dtype=object)

## Cuarta etapa: chequeamos duplicados, nulos y guardamos

In [184]:
df_unificado.isna().sum()

FECHA                0
DESDE                0
HASTA                0
HORA                 0
LINEA                0
ESTACION             0
BOCA                 0
MOLINETE             0
MOLINETE_ORIGINAL    0
PAX_TOTAL            0
dtype: int64

In [185]:
df_unificado[df_unificado.duplicated()]

,FECHA,DESDE,HASTA,HORA,LINEA,ESTACION,BOCA,MOLINETE,MOLINETE_ORIGINAL,PAX_TOTAL
43902725,2025-02-08,09:00,09:15,9,C,RETIRO,RETIRO,TURN01,LineaC_Retiro_Turn01,1
43903617,2025-02-08,11:15,11:30,11,C,RETIRO,RETIRO,TURN01,LineaC_Retiro_Turn01,3
43904356,2025-02-08,13:00,13:15,13,C,RETIRO,RETIRO,TURN01,LineaC_Retiro_Turn01,4
43906190,2025-02-08,17:15,17:30,17,C,RETIRO,RETIRO,TURN01,LineaC_Retiro_Turn01,5
43908045,2025-02-08,21:45,22:00,21,C,RETIRO,RETIRO,TURN01,LineaC_Retiro_Turn01,1
...,...,...,...,...,...,...,...,...,...,...
48593617,2025-12-08,20:45,21:00,20,E,RETIRO E,RETIROE_N,TURN02,LineaE_RetiroE_N_Turn02,1
48593691,2025-12-08,21:00,21:15,21,E,RETIRO E,RETIROE_N,TURN01,LineaE_RetiroE_N_Turn01,1
48593798,2025-12-08,21:15,21:30,21,D,CALLAO,CALLAOD_N,TURN01,LineaD_CallaoD_N_Turn01,2
48593975,2025-12-08,21:45,22:00,21,E,RETIRO E,RETIROE_N,TURN02,LineaE_RetiroE_N_Turn02,1


In [186]:
df.drop_duplicates(inplace=True)

Volvemos a fraccionar por año, y guardamos:

In [ ]:
output_dir = "../output/22-26"
os.makedirs(output_dir, exist_ok=True)

for anio, df_grupo in df_unificado.groupby(df_unificado['FECHA'].dt.year):
    ruta_archivo = os.path.join(output_dir, f"df{anio}.parquet")
    
    df_grupo.to_parquet(
        ruta_archivo, 
        index=False, 
        engine='pyarrow', 
        compression='snappy',
        coerce_timestamps='ms'
    )
    print(f"Año {anio} guardado exitosamente ({len(df_grupo):,} filas).")

Año 2022 guardado exitosamente (12,136,381 filas).
Año 2023 guardado exitosamente (12,047,696 filas).
Año 2024 guardado exitosamente (11,440,437 filas).
Año 2025 guardado exitosamente (13,132,605 filas).
Año 2026 guardado exitosamente (7,085,791 filas).
